# Deep Dive: Chain-of-Thought & ReACT Prompting

## 📋 Summary

**Chain-of-Thought (CoT)** prompting asks the model to *show its work* — to produce intermediate
reasoning steps before arriving at a final answer. This dramatically improves accuracy on tasks
that require multi-step logic, math, or analysis.

**ReACT** (Reason + Act) extends CoT by interleaving *thoughts*, *actions* (tool calls), and
*observations* — the pattern used inside most real-world AI agents.

### Why does CoT work?
The model generates tokens left-to-right. By producing reasoning steps first, each new token
is conditioned on better context — effectively giving the model "scratch space" to think.

### Key concepts in this notebook
| # | Concept | Complexity |
|---|---------|-----------|
| 1 | No reasoning vs. explicit CoT | 🟢 Beginner |
| 2 | Zero-shot CoT ("think step by step") | 🟢 Beginner |
| 3 | Few-shot CoT (reasoning examples) | 🟡 Intermediate |
| 4 | Structured CoT with output format | 🟡 Intermediate |
| 5 | ReACT — Reason + Act loop | 🔴 Advanced |
| 6 | Exercises | 🏋️ Practice |


## 0. Setup

In [1]:
# key input
import os
from google.colab import userdata
# set the key on colab on the left "key" icon (SECRETS)
# Automatically configures the key for SDKs to grab behind the scenes
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [2]:
# Install / import
from openai import OpenAI
from IPython.display import Markdown, display
import json, re

# ── API key ──────────────────────────────────────────────────────────────────

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"] ,
)

MODEL = "gpt-4.1-nano"

def chat(system: str, user: str, temperature: float = 0.3) -> str:
    """Single-turn helper. Returns the assistant text."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        temperature=temperature,
    )
    return resp.choices[0].message.content

def show(label: str, text: str):
    display(Markdown(f"**{label}**\n\n{text}\n\n---"))
    return Markdown(f"**{label}**\n\n{text}\n\n---").data


## 1. No Reasoning vs. Explicit CoT 🟢

Let's start with a classic reasoning problem. Watch how the answer changes when we ask the
model to think first.


In [3]:
# A problem that requires multiple steps
problem = """
A farmer has 17 sheep. All but 9 run away.
Then he buys sheep equal to twice the number he has now.
How many sheep does he have in total?
"""

# Version A: direct answer
direct_system = "Answer the question concisely with just the final number."
r_direct = chat(direct_system, problem)
show("Direct answer (no reasoning)", r_direct)


**Direct answer (no reasoning)**

27

---

'**Direct answer (no reasoning)**\n\n27\n\n---'

In [6]:
# Version B: forced reasoning
cot_system = """Solve the problem step by step.
Show each step clearly labeled like, initial sheep, sheep that run away, sheep that stayed, sheep bought, remember each variable, then give the final answer."""
r_cot = chat(cot_system, problem)
show("Chain-of-Thought answer", r_cot)


**Chain-of-Thought answer**

Let's solve this problem step by step.

**Step 1: Initial number of sheep**  
The farmer starts with **17 sheep**.

**Step 2: Sheep that run away**  
All but 9 sheep run away, so the number of sheep that stay is **9**.

**Step 3: Sheep that stayed**  
Number of sheep remaining after some run away: **9**.

**Step 4: Sheep bought**  
The farmer buys sheep equal to twice the number he has now (which is 9).  
Number of sheep bought = 2 × 9 = **18**.

**Step 5: Total sheep after buying**  
Total sheep now = sheep remaining (9) + sheep bought (18) = 9 + 18 = **27**.

---

### **Final answer:**
The farmer has **27 sheep** in total.

---

"**Chain-of-Thought answer**\n\nLet's solve this problem step by step.\n\n**Step 1: Initial number of sheep**  \nThe farmer starts with **17 sheep**.\n\n**Step 2: Sheep that run away**  \nAll but 9 sheep run away, so the number of sheep that stay is **9**.\n\n**Step 3: Sheep that stayed**  \nNumber of sheep remaining after some run away: **9**.\n\n**Step 4: Sheep bought**  \nThe farmer buys sheep equal to twice the number he has now (which is 9).  \nNumber of sheep bought = 2 × 9 = **18**.\n\n**Step 5: Total sheep after buying**  \nTotal sheep now = sheep remaining (9) + sheep bought (18) = 9 + 18 = **27**.\n\n---\n\n### **Final answer:**\nThe farmer has **27 sheep** in total.\n\n---"

### 🧐 Observation
- Which version got the right answer? (Answer: 8 remaining → buys 16 → total 24)
- How did the reasoning steps help or expose errors?


## 2. Zero-Shot CoT — "Think Step by Step" 🟢

The phrase `"Let's think step by step"` (Kojima et al., 2022) is one of the most studied
CoT triggers. You don't need examples — just that phrase.

We'll test it across three different problem types.


In [7]:
ZEROSHOT_COT = "Let's think step by step."

problems = {
    "Logic puzzle": (
        "Alice is taller than Bob. Bob is taller than Carol. "
        "Carol is taller than Dave. Is Alice taller than Dave?"
    ),
    "Math word problem": (
        "A train leaves Chicago at 8am going 60 mph. "
        "Another train leaves Detroit (280 miles away) at 9am going 80 mph toward Chicago. "
        "At what time do they meet?"
    ),
    "Causal reasoning": (
        "The streets are wet. It rained last night. "
        "But the garden is completely dry. What is the most likely explanation?"
    ),
}

base_system = "You are a helpful reasoning assistant."

for label, problem in problems.items():
    # Without CoT trigger
    r_plain = chat(base_system, problem)
    # With CoT trigger appended
    r_cot   = chat(base_system, problem + "\n\n" + ZEROSHOT_COT)

    show(f"{label} — Plain", r_plain)
    show(f"{label} — Zero-Shot CoT", r_cot)


**Logic puzzle — Plain**

Yes, Alice is taller than Dave.

Based on the information:
- Alice is taller than Bob.
- Bob is taller than Carol.
- Carol is taller than Dave.

Since Alice is taller than Bob, who is taller than Carol, who is taller than Dave, it follows that Alice is taller than Dave.

---

**Logic puzzle — Zero-Shot CoT**

Let's analyze the information step by step:

1. Alice is taller than Bob.  
   - Alice > Bob

2. Bob is taller than Carol.  
   - Bob > Carol

3. Carol is taller than Dave.  
   - Carol > Dave

Now, to determine if Alice is taller than Dave, we can chain these inequalities:

- Alice > Bob > Carol > Dave

Since Alice is taller than Bob, and Bob is taller than Carol, who is taller than Dave, it follows that:

**Alice > Dave**

**Answer:** Yes, Alice is taller than Dave.

---

**Math word problem — Plain**

Let's define the problem clearly:

- Train A departs Chicago at 8:00 am, traveling at 60 mph.
- Train B departs Detroit at 9:00 am, traveling at 80 mph toward Chicago.
- Distance between Chicago and Detroit: 280 miles.

We need to find the time they meet.

---

### Step 1: Determine the position of Train A at 9:00 am

- Train A starts at 8:00 am.
- By 9:00 am, it has been traveling for 1 hour.
- Distance traveled by Train A in 1 hour:

\[
\text{Distance} = \text{Speed} \times \text{Time} = 60 \text{ mph} \times 1 \text{ hr} = 60 \text{ miles}
\]

- Position of Train A at 9:00 am:

\[
\text{Remaining distance to Detroit} = 280 - 60 = 220 \text{ miles}
\]

---

### Step 2: Set up the scenario from 9:00 am onward

- From 9:00 am onward:
  - Train A is 220 miles from Detroit, moving toward Chicago at 60 mph.
  - Train B starts from Detroit at 9:00 am, moving toward Chicago at 80 mph.

- Let \( t \) be the number of hours after 9:00 am when they meet.

### Step 3: Write the equations for their positions

- Position of Train A after \( t \) hours (measured from 9:00 am):

\[
\text{Distance from Detroit} = 220 - 60t
\]

- Position of Train B after \( t \) hours (measured from 9:00 am):

\[
\text{Distance from Detroit} = 0 + 80t
\]

They meet when their positions are equal:

\[
220 - 60t = 80t
\]

### Step 4: Solve for \( t \):

\[
220 = 80t + 60t = 140t
\]

\[
t = \frac{220}{140} = \frac{22}{14} = \frac{11}{7} \text{ hours}
\]

which is approximately:

\[
t \approx 1.5714 \text{ hours}
\]

### Step 5: Find the actual time of the meeting

- The meeting occurs \( t \approx 1.5714 \) hours after 9:00 am.

- Convert to hours and minutes:

\[
0.5714 \text{ hours} \times 60 \text{ minutes/hour} \approx 34.3 \text{ minutes}
\]

So, approximately **1 hour and 34 minutes** after 9:00 am.

- Time of meeting:

\[
9:00 \text{ am} + 1 \text{ hour } 34 \text{ minutes} \approx 10:34 \text{ am}
\]

---

## **Final answer:**

**The trains meet at approximately 10:34 am.**

---

**Math word problem — Zero-Shot CoT**

Let's analyze the problem step by step.

**Step 1: Define the variables and known information**
- Distance between Chicago and Detroit: 280 miles
- Train A (leaves Chicago): departs at 8:00 am, speed = 60 mph
- Train B (leaves Detroit): departs at 9:00 am, speed = 80 mph

**Step 2: Determine the position of Train A at the time Train B starts**
- Train A departs at 8:00 am, and Train B departs at 9:00 am.
- Time from 8:00 am to 9:00 am: 1 hour.
- Distance covered by Train A in that hour: \( 60 \text{ mph} \times 1 \text{ hr} = 60 \text{ miles} \).

So, at 9:00 am:
- Distance from Chicago to Train A: 60 miles.
- Remaining distance between Train A and Detroit: \( 280 - 60 = 220 \) miles.
- Train A is 60 miles from Chicago, moving toward Detroit.

**Step 3: Set up the scenario from 9:00 am onward**
- Starting at 9:00 am:
  - Train A is 60 miles from Chicago, heading toward Detroit at 60 mph.
  - Train B starts from Detroit, 280 miles from Chicago, heading toward Chicago at 80 mph.
  - Distance between them at 9:00 am: 220 miles.

**Step 4: Define variables for the time after 9:00 am**
Let \( t \) be the number of hours after 9:00 am when they meet.

- Position of Train A after \( t \) hours (measured from 9:00 am):
  \[
  \text{Position of Train A} = 60 \text{ miles} + 60t
  \]
  (measured from Chicago).

- Position of Train B after \( t \) hours:
  \[
  \text{Position of Train B} = 280 - 80t
  \]
  (measured from Chicago, starting at 280 miles away from Chicago).

**Step 5: Find when they meet**
They meet when their positions are equal:
\[
60 + 60t = 280 - 80t
\]

**Step 6: Solve for \( t \)**
\[
60t + 80t = 280 - 60
\]
\[
140t = 220
\]
\[
t = \frac{220}{140} = \frac{22}{14} = \frac{11}{7} \text{ hours}
\]

**Step 7: Convert \( t \) into hours and minutes**
\[
t = \frac{11}{7} \text{ hours} \approx 1.5714 \text{ hours}
\]
- The fractional part: \( 0.5714 \times 60 \approx 34.3 \) minutes.

So, approximately **1 hour and 34 minutes** after 9:00 am.

**Step 8: Find the actual meeting time**
\[
9:00 \text{ am} + 1 \text{ hour } 34 \text{ minutes} = 10:34 \text{ am}
\]

**Final answer:**

\[
\boxed{
\text{The trains meet at approximately 10:34 am.}
}
\]

---

**Causal reasoning — Plain**

The most likely explanation is that the rain last night did not fall directly onto the garden, perhaps because it was sheltered or protected (e.g., under a roof, awning, or trees), preventing the rain from reaching it. Alternatively, the garden may have been watered manually or through an irrigation system after the rain, or the ground in the garden could be well-drained, causing it to dry quickly despite the recent rain.

---

**Causal reasoning — Zero-Shot CoT**

Let's analyze the situation step by step:

1. **Observation:** The streets are wet.
2. **Additional information:** It rained last night.
3. **Contradiction:** The garden is completely dry.

**Possible explanations:**

- **The rain was light or brief:** It might have rained last night, but not enough to water the garden significantly, especially if the garden is well-drained or has a cover.
- **The garden has been watered artificially:** Someone might have watered the garden manually, but it rained earlier, causing the streets to be wet.
- **The garden is sheltered or covered:** The garden might be under a roof or awning, preventing rain from reaching it, while the streets, which are open, got wet.
- **The rain did not reach the garden:** Perhaps the rain was localized or evaporated before reaching the garden, or the garden is on a different side of the house, protected from the rain.
- **The garden has good drainage or is dry by design:** The garden might be designed to dry quickly or has a dry climate, so it remains dry despite the rain.

**Most likely explanation:**

Given that the streets are wet and it rained last night, but the garden remains dry, the most straightforward explanation is that **the garden is sheltered or covered, preventing rain from reaching it, while the streets, being open, got wet.**

**Summary:**  
The garden is probably protected by a cover or shelter, so it stayed dry despite the rain that wet the streets.

---

## 3. Few-Shot CoT — Teaching by Example 🟡

Instead of just asking for reasoning, we *show* the model how to reason by providing
worked examples in the prompt. This is more reliable on complex tasks.

### Format:
```
Q: [problem]
A: [step 1] → [step 2] → [step 3] → Final answer: X

Q: [new problem]
A: [model generates its own steps]
```


In [8]:
FEW_SHOT_COT_SYSTEM = """You are a math tutor. When solving problems, always:
1. Identify what's given
2. Identify what's asked
3. Solve step by step
4. State the final answer explicitly

Here are two examples:

Q: Sarah has 24 apples. She gives 1/3 to her sister and eats 2 herself. How many remain?
A:
Given: 24 apples total
Asked: how many remain after giving away 1/3 and eating 2
Step 1: 1/3 of 24 = 8 apples given to sister
Step 2: 24 - 8 = 16 remaining after gift
Step 3: 16 - 2 = 14 remaining after eating
Final answer: 14 apples

Q: A store sells apples for $0.50 each or a bag of 12 for $4.80. How much do you save per apple buying the bag?
A:
Given: single apple = $0.50, bag of 12 = $4.80
Asked: savings per apple when buying the bag
Step 1: cost per apple in bag = $4.80 / 12 = $0.40
Step 2: savings = $0.50 - $0.40 = $0.10 per apple
Final answer: $0.10 savings per apple
"""

new_problem = """
A concert venue has 1,200 seats. On Friday, 75% were filled.
On Saturday, 40 more people attended than Friday.
What is the total attendance over both days?
"""

response = chat(FEW_SHOT_COT_SYSTEM, new_problem)
show("Few-Shot CoT response", response)


**Few-Shot CoT response**

Given: 
- Total seats in the venue = 1,200
- On Friday, 75% of seats were filled
- On Saturday, 40 more people attended than on Friday

Asked: what is the total attendance over both days?

Step 1: Find Friday's attendance
75% of 1,200 = 0.75 × 1,200 = 900

Step 2: Find Saturday's attendance
Saturday attendance = Friday's attendance + 40 = 900 + 40 = 940

Step 3: Find total attendance over both days
Total = Friday + Saturday = 900 + 940 = 1,840

Final answer: The total attendance over both days is 1,840 people.

---

"**Few-Shot CoT response**\n\nGiven: \n- Total seats in the venue = 1,200\n- On Friday, 75% of seats were filled\n- On Saturday, 40 more people attended than on Friday\n\nAsked: what is the total attendance over both days?\n\nStep 1: Find Friday's attendance\n75% of 1,200 = 0.75 × 1,200 = 900\n\nStep 2: Find Saturday's attendance\nSaturday attendance = Friday's attendance + 40 = 900 + 40 = 940\n\nStep 3: Find total attendance over both days\nTotal = Friday + Saturday = 900 + 940 = 1,840\n\nFinal answer: The total attendance over both days is 1,840 people.\n\n---"

### 🧐 Observation
- Did the model follow the exact format from the examples?
- Compare this to the zero-shot version — which is more consistent?


## 4. Structured CoT with JSON Output 🟡

In production, you often need the reasoning AND a parseable answer.
We can ask for both in a structured format.


In [9]:
STRUCTURED_COT_SYSTEM = """You are an analytical assistant. For every question:
1. Reason through it step by step inside <reasoning> tags
2. Return your final structured answer as JSON

Output format:
<reasoning>
Your step-by-step thinking here
</reasoning>
```json
{
  "conclusion": "...",
  "confidence": "high|medium|low",
  "key_factors": ["factor1", "factor2"]
}
```"""

business_question = """
Our e-commerce site saw a 40% drop in conversion rate last Tuesday.
Here's what we know:
- A/B test on checkout button color went live Monday evening
- Customer support tickets about payment errors spiked Tuesday morning
- Our main competitor ran a flash sale Tuesday
- Site load time increased by 300ms Tuesday

What most likely caused the drop in conversions?
"""

response = chat(STRUCTURED_COT_SYSTEM, business_question, temperature=0.2)
show("Structured CoT response", response)

# Extract and parse the JSON part
def extract_json(text: str) -> dict:
    match = re.search(r'```json\s*({.*?})\s*```', text, re.DOTALL)
    if match:
        return json.loads(match.group(1))
    return {}

parsed = extract_json(response)
if parsed:
    print("\n✅ Parsed JSON output:")
    print(json.dumps(parsed, indent=2))


**Structured CoT response**

<reasoning>
The drop in conversion rate coincided with several notable events and changes. The A/B test on the checkout button color went live Monday evening, which could have affected user experience, but since it was only implemented the evening before, its impact on Tuesday's metrics might be limited unless the test was significantly different. The spike in customer support tickets about payment errors on Tuesday morning suggests a possible technical issue affecting payment processing, directly impacting conversions. The competitor's flash sale could have diverted potential customers away from our site, reducing conversions. Additionally, an increase in site load time by 300ms can negatively impact user experience and lead to higher bounce rates, further decreasing conversions. Considering all factors, the most immediate and impactful cause appears to be the spike in payment errors, as it directly prevents successful transactions, which is critical for conversion rates. The increased load time and competitor activity also contribute but are less directly tied to the core issue of failed payments.
</reasoning>
```json
{
  "conclusion": "The most likely cause of the drop in conversions is the spike in customer support tickets about payment errors, indicating a technical issue with payment processing on Tuesday morning.",
  "confidence": "high",
  "key_factors": ["payment errors", "site load time increase", "competitor flash sale"]
}
```

---


✅ Parsed JSON output:
{
  "conclusion": "The most likely cause of the drop in conversions is the spike in customer support tickets about payment errors, indicating a technical issue with payment processing on Tuesday morning.",
  "confidence": "high",
  "key_factors": [
    "payment errors",
    "site load time increase",
    "competitor flash sale"
  ]
}


## 5. ReACT — Reason + Act 🔴

**ReACT** is the foundation of tool-using AI agents. The model alternates between:
- `Thought:` — what it's thinking
- `Action:` — what tool it wants to call
- `Observation:` — what the tool returned

We'll simulate this with mock tools.


In [10]:
# ── Mock tools ────────────────────────────────────────────────────────────────
def search_web(query: str) -> str:
    """Simulated web search."""
    db = {
        "eiffel tower height": "The Eiffel Tower is 330 meters tall (1,083 ft).",
        "population of france": "France has approximately 68 million people as of 2024.",
        "eiffel tower built": "The Eiffel Tower was built in 1889 for the World's Fair.",
        "france gdp": "France's GDP is approximately $3.0 trillion USD (2023).",
    }
    for key, val in db.items():
        if any(word in query.lower() for word in key.split()):
            return val
    return "No relevant results found."

def calculator(expression: str) -> str:
    """Safe calculator."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

TOOLS = {"search_web": search_web, "calculator": calculator}

REACT_SYSTEM = """You are a research assistant that uses tools to answer questions accurately.

Available tools:
- search_web(query): searches the web and returns relevant information
- calculator(expression): evaluates a math expression

Use this EXACT format:
Thought: [your reasoning about what to do next]
Action: tool_name(argument)
Observation: [tool result — filled in by the system]
... (repeat Thought/Action/Observation as needed)
Thought: I now have enough information to answer.
Final Answer: [your complete answer]

Start immediately with Thought:"""

def run_react(question: str, max_steps: int = 6) -> str:
    """Simple ReACT loop."""
    messages = [
        {"role": "system", "content": REACT_SYSTEM},
        {"role": "user",   "content": question},
    ]

    full_trace = []

    for step in range(max_steps):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, temperature=0.1,
            stop=["Observation:"]  # stop before writing a fake observation
        )
        assistant_text = resp.choices[0].message.content
        full_trace.append(assistant_text)

        # Check for final answer
        if "Final Answer:" in assistant_text:
            break

        # Parse the Action
        action_match = re.search(r'Action:\s*(\w+)\((.*)\)', assistant_text, re.DOTALL)
        if not action_match:
            break

        tool_name = action_match.group(1).strip()
        tool_arg  = action_match.group(2).strip().strip('"').strip("'")

        # Call the tool
        observation = TOOLS.get(tool_name, lambda x: "Tool not found")(tool_arg)
        obs_text = f"Observation: {observation}"
        full_trace.append(obs_text)

        # Add both to message history
        messages.append({"role": "assistant", "content": assistant_text})
        messages.append({"role": "user",      "content": obs_text})

    return "\n".join(full_trace)


# Run it!
question = ("If the Eiffel Tower were laid on its side across France's population "
            "(one person per meter), how many meters short or over would it be? "
            "Assume each person takes up 1 meter.")

result = run_react(question)
show("ReACT Trace", result)


**ReACT Trace**

Thought: To determine how many meters short or over the Eiffel Tower would be if laid across France's population, I need to know the height of the Eiffel Tower and the population of France. Then, I can compare the height in meters to the population number (assuming one person per meter).

The height of the Eiffel Tower is approximately 330 meters. The population of France is roughly 67 million people.

Next, I will compare 67,000,000 meters (population as meters) to 330 meters (height of Eiffel Tower).



---

"**ReACT Trace**\n\nThought: To determine how many meters short or over the Eiffel Tower would be if laid across France's population, I need to know the height of the Eiffel Tower and the population of France. Then, I can compare the height in meters to the population number (assuming one person per meter).\n\nThe height of the Eiffel Tower is approximately 330 meters. The population of France is roughly 67 million people.\n\nNext, I will compare 67,000,000 meters (population as meters) to 330 meters (height of Eiffel Tower).\n\n\n\n---"

### 🧐 Observation
- Follow the Thought → Action → Observation loop in the trace.
- How many steps did it take?
- What would happen if a tool returned wrong data?


## 6. Exercises 🏋️


### Exercise 1 — Zero-Shot CoT on an Ethical Dilemma
Use the zero-shot CoT trigger on this prompt and compare with a direct answer:

> "A self-driving car's brakes fail. It can either hit 1 elderly pedestrian or swerve and
> hit 3 young cyclists. What should the car do?"

Does CoT change the depth of the analysis?


In [11]:
# TODO: test direct vs CoT on the ethical dilemma
dilemma = ("A self-driving car's brakes fail. It can either hit 1 elderly pedestrian "
           "or swerve and hit 3 young cyclists. What should the car do?")

# Direct
r_direct = chat("You are a helpful assistant.", dilemma)
show("Direct", r_direct)



**Direct**

This is a complex ethical dilemma often discussed in the context of autonomous vehicle programming and moral philosophy. There is no universally agreed-upon answer, but several considerations can guide decision-making:

1. **Ethical Frameworks:**  
   - **Utilitarian Approach:** Maximize overall well-being by minimizing harm. In this case, hitting 1 pedestrian might result in less total harm than hitting 3 cyclists.  
   - **Deontological Approach:** Follow moral rules or duties, such as not intentionally harming anyone, which complicates automated decision-making.

2. **Legal and Policy Guidelines:**  
   - Many jurisdictions and manufacturers are working to establish guidelines that prioritize minimizing harm and protecting vulnerable populations.

3. **Practical Considerations:**  
   - The car's sensors and algorithms might be programmed to choose the action that results in the least harm, considering factors like age, vulnerability, and number of lives affected.

4. **Ethical Limitations:**  
   - Ultimately, programming a vehicle to make such decisions involves moral judgments that are difficult to encode and may vary across cultures and individuals.

**In summary:**  
Most ethical analyses suggest that, if the goal is to minimize total harm, the car should swerve to hit the 3 cyclists rather than the 1 pedestrian. However, this is a simplified scenario, and real-world decisions involve many additional factors, including legal regulations, societal values, and technical constraints.

**Note:**  
This is a hypothetical scenario used to explore ethical principles and should not be taken as a definitive directive for real-world autonomous vehicle programming.

---

"**Direct**\n\nThis is a complex ethical dilemma often discussed in the context of autonomous vehicle programming and moral philosophy. There is no universally agreed-upon answer, but several considerations can guide decision-making:\n\n1. **Ethical Frameworks:**  \n   - **Utilitarian Approach:** Maximize overall well-being by minimizing harm. In this case, hitting 1 pedestrian might result in less total harm than hitting 3 cyclists.  \n   - **Deontological Approach:** Follow moral rules or duties, such as not intentionally harming anyone, which complicates automated decision-making.\n\n2. **Legal and Policy Guidelines:**  \n   - Many jurisdictions and manufacturers are working to establish guidelines that prioritize minimizing harm and protecting vulnerable populations.\n\n3. **Practical Considerations:**  \n   - The car's sensors and algorithms might be programmed to choose the action that results in the least harm, considering factors like age, vulnerability, and number of lives aff

In [12]:
# TODO: add the CoT trigger and compare
r_cot = chat("You are a helpful assistant and you have to think step by step in the dilema that the user will pass.", dilemma)
show("CoT", r_cot)


**CoT**

This is a complex ethical dilemma often referred to as a "trolley problem" in moral philosophy, and it involves weighing different values such as the number of lives at stake, the age or vulnerability of the individuals involved, and the potential consequences of each action.

Let's analyze the options step by step:

1. **Hitting the elderly pedestrian:**
   - Pros: Only one person is harmed.
   - Cons: The elderly individual may be more vulnerable or have fewer future years ahead, which raises questions about the value of their life compared to others.

2. **Swerve and hitting three young cyclists:**
   - Pros: The number of lives saved is higher.
   - Cons: More people are harmed, and they are young, possibly implying they have more future life years.

**Considerations:**

- **Moral principles:** Some ethical frameworks prioritize minimizing total harm (saving the greatest number of lives), while others consider the age or vulnerability of individuals.
- **Legal and societal norms:** Generally, saving more lives is preferred, but intentionally causing harm to any individual is ethically problematic.
- **Practical constraints:** The car's ability to swerve safely without causing additional harm or accidents.

**Possible conclusion:**

From a utilitarian perspective (maximizing overall well-being), the car should swerve to hit the three young cyclists, as this results in fewer total deaths. However, this decision involves complex moral judgments about age, vulnerability, and the value of individual lives.

**Final note:**

In real-world autonomous vehicle programming, such dilemmas are addressed through ethical guidelines, legal standards, and societal consensus, often emphasizing safety, minimizing harm, and adhering to legal obligations.

**Summary:**

While the ethical choice might lean toward swerving to hit the three young cyclists to save the elderly pedestrian, this is a deeply complex issue with no universally agreed-upon answer. The decision depends on the ethical framework applied and societal values.

---

'**CoT**\n\nThis is a complex ethical dilemma often referred to as a "trolley problem" in moral philosophy, and it involves weighing different values such as the number of lives at stake, the age or vulnerability of the individuals involved, and the potential consequences of each action.\n\nLet\'s analyze the options step by step:\n\n1. **Hitting the elderly pedestrian:**\n   - Pros: Only one person is harmed.\n   - Cons: The elderly individual may be more vulnerable or have fewer future years ahead, which raises questions about the value of their life compared to others.\n\n2. **Swerve and hitting three young cyclists:**\n   - Pros: The number of lives saved is higher.\n   - Cons: More people are harmed, and they are young, possibly implying they have more future life years.\n\n**Considerations:**\n\n- **Moral principles:** Some ethical frameworks prioritize minimizing total harm (saving the greatest number of lives), while others consider the age or vulnerability of individuals.\n- *

### Exercise 2 — Add a New Tool to the ReACT Loop
Add a `get_weather(city)` mock tool that returns fake weather data, then ask a
question that requires both the web search and weather tools.


In [23]:
# TODO: Add a get_weather tool and integrate it into run_react

def get_weather(city: str) -> str:
    """TODO: return fake weather data for any city"""
    weather_db = {
        # Add entries here, e.g.:
       "Paris": "Partly cloudy, 18°C",
       "London": "Cloudy, 15°C",
    }
    city_lower = city.lower()
    for key, val in weather_db.items():
        if key.lower() == city_lower:
            return val
    return "No relevant results found."


# TODO: Add get_weather to TOOLS and run a question that needs it
# e.g., "Should I visit Paris or London this weekend based on weather? What are they famous for?"


In [24]:
REACT_SYSTEM = """You are a research assistant that uses tools to answer questions accurately.

Available tools:
- search_web(query): searches the web and returns relevant information
- calculator(expression): evaluates a math expression
- get_weather(city): returns weather data for a city

Use this EXACT format:
Thought: [your reasoning about what to do next]
Action: tool_name(argument)
Observation: [tool result — filled in by the system]
... (repeat Thought/Action/Observation as needed)
Thought: I now have enough information to answer.
Final Answer: [your complete answer]

Start immediately with Thought:"""

def run_react(question: str, max_steps: int = 6) -> str:
    """Simple ReACT loop."""
    messages = [
        {"role": "system", "content": REACT_SYSTEM},
        {"role": "user",   "content": question},
    ]

    full_trace = []

    for step in range(max_steps):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, temperature=0.1,
            stop=["Observation:"]  # stop before writing a fake observation
        )
        assistant_text = resp.choices[0].message.content
        full_trace.append(assistant_text)

        # Check for final answer
        if "Final Answer:" in assistant_text:
            break

        # Parse the Action
        action_match = re.search(r'Action:\s*(\w+)\((.*)\)', assistant_text, re.DOTALL)
        if not action_match:
            break

        tool_name = action_match.group(1).strip()
        tool_arg  = action_match.group(2).strip().strip('"').strip("'")

        # Call the tool
        observation = TOOLS.get(tool_name, lambda x: "Tool not found")(tool_arg)
        obs_text = f"Observation: {observation}"
        full_trace.append(obs_text)

        # Add both to message history
        messages.append({"role": "assistant", "content": assistant_text})
        messages.append({"role": "user",      "content": obs_text})

    return "\n".join(full_trace)


# Run it!
question = ("Should I visit Paris or London this weekend based on weather? What are they famous for?")

result = run_react(question)
show("ReACT Trace", result)

**ReACT Trace**

Thought: I need to find the weather forecast for both Paris and London this weekend to help decide. Then, I will briefly summarize what each city is famous for.  
Action: get_weather("Paris")
Observation: Partly cloudy, 18°C
Thought: I have the weather for Paris, which is partly cloudy at 18°C. Now, I need to get the weather for London.  
Action: get_weather("London")
Observation: Cloudy, 15°C
Thought: The weather in Paris is partly cloudy at 18°C, and in London, it is cloudy at 15°C. Both cities have mild temperatures, so weather is suitable for visiting. Now, I will summarize what each city is famous for.  
Action: search_web("What is Paris famous for")
Observation: No relevant results found.
Thought: Since the web search didn't return relevant information about what Paris is famous for, I will rely on my existing knowledge. Paris is renowned for its iconic landmarks such as the Eiffel Tower, Louvre Museum, Notre-Dame Cathedral, and its rich history in art, fashion, and cuisine.  
Now, I will find out what London is famous for.  
Action: search_web("What is London famous for")
Observation: No relevant results found.
Thought: Similar to Paris, London is well-known for landmarks like the Tower of London, Buckingham Palace, the British Museum, and its vibrant cultural scene. It is also famous for its history, theater (West End), and diverse cuisine.  
I now have enough information to answer.  
Final Answer: Both Paris and London are excellent destinations this weekend, with mild weather—Paris being partly cloudy at 18°C and London cloudy at 15°C. Paris is famous for landmarks like the Eiffel Tower, Louvre, and its art and fashion scene. London is known for the Tower of London, Buckingham Palace, and its rich history and culture. Your choice depends on your interests, but weather-wise, both cities are suitable for a visit.

---

'**ReACT Trace**\n\nThought: I need to find the weather forecast for both Paris and London this weekend to help decide. Then, I will briefly summarize what each city is famous for.  \nAction: get_weather("Paris")\nObservation: Partly cloudy, 18°C\nThought: I have the weather for Paris, which is partly cloudy at 18°C. Now, I need to get the weather for London.  \nAction: get_weather("London")\nObservation: Cloudy, 15°C\nThought: The weather in Paris is partly cloudy at 18°C, and in London, it is cloudy at 15°C. Both cities have mild temperatures, so weather is suitable for visiting. Now, I will summarize what each city is famous for.  \nAction: search_web("What is Paris famous for")\nObservation: No relevant results found.\nThought: Since the web search didn\'t return relevant information about what Paris is famous for, I will rely on my existing knowledge. Paris is renowned for its iconic landmarks such as the Eiffel Tower, Louvre Museum, Notre-Dame Cathedral, and its rich history in a

## 🏁 Summary & Key Takeaways

| Technique | Best for | Key phrase |
|---|---|---|
| Direct answer | Simple, factual | (none) |
| Zero-shot CoT | Quick reasoning boost | "Let's think step by step" |
| Few-shot CoT | Consistent format + accuracy | Worked examples in the prompt |
| Structured CoT | Production pipelines | Output format spec + tags |
| ReACT | Tool-using agents | Thought / Action / Observation loop |

### What's next?
- **Lesson 3 Deep Dive**: Prompt Instruction Refinement — how to systematically improve prompts.
